# Entity Resolution & Product Grouping

**Project:** StyleHaus + UrbanThread E-Commerce Product Catalog Consolidation  
**Platform:** Microsoft Fabric Lakehouse / PySpark

## Objective
Create one product group for each approved cross-source `MATCH` pair and an individual group for every other Silver product. `REVIEW` and `NO_MATCH` records are not consolidated.

## Existing Lakehouse tables
- `silver.products`
- `semantic.finalmatchingdecisions`

## Output
An in-memory DataFrame named `ProductGroupMembersDf` with:
- `GoldProductGroupId`
- `SourceSystem`
- `SourceProductId`
- `GroupType`

This sub-phase prepares the grouping map. It does **not** write the final Gold tables; survivorship and field-level lineage are implemented in subsequent Phase 4 sub-phases.

## Grouping rules

1. Use `FinalMatchDecision_V2 = 'MATCH'` only.
2. A StyleHaus product and its approved UrbanThread counterpart share one group ID.
3. All products not included in an approved pair get their own group ID. This includes products whose pair decision is `REVIEW` or `NO_MATCH`.
4. Group IDs are deterministic SHA-256 hashes of the source-qualified product identifiers.
5. This implementation enforces one-to-one approved matches. If the matching input contains conflicting assignments, it stops rather than silently choosing a pair.

In [1]:
# Cell 1 – Imports and input tables

from pyspark.sql import functions as F

SilverDf = spark.table("silver.products")

MatchingDf = spark.table("semantic.finalmatchingdecisions")

print("Silver records:", SilverDf.count())
print("Matching decision records:", MatchingDf.count())

StatementMeta(, 650a0266-f6d3-41c0-bb82-b95dff4eb5a1, 3, Finished, Available, Finished, False)

Silver records: 164
Matching decision records: 185


In [2]:
# Cell 2 – Select approved MATCH pairs

ApprovedMatchesDf = (
    MatchingDf
    .filter(F.upper(F.trim(F.col("FinalMatchDecision_V2"))) == "MATCH")
    .select(
        F.trim(F.col("StyleHaus_ProductID")).alias("StyleHaus_ProductID"),
        F.trim(F.col("UrbanThread_ProductID")).alias("UrbanThread_ProductID")
    )
    .dropDuplicates()
)

print("Approved matching pairs:", ApprovedMatchesDf.count())
display(ApprovedMatchesDf)

StatementMeta(, 650a0266-f6d3-41c0-bb82-b95dff4eb5a1, 4, Finished, Available, Finished, False)

Approved matching pairs: 22


SynapseWidget(Synapse.DataFrame, 668d5073-1b21-4041-9661-6aa2c5f3ca3d)

## Integrated integrity guard

The following checks are part of the grouping implementation. They protect the group construction from invalid keys, duplicate assignments, and approved match IDs that cannot be found in the corresponding Silver source.

In [6]:
# Cell 3 – Enforce valid one-to-one matching inputs

RequiredSilverColumns = {"SourceSystem", "SourceProductID"}
RequiredMatchingColumns = {
    "FinalMatchDecision_V2",
    "StyleHaus_ProductID",
    "UrbanThread_ProductID"
}

MissingSilverColumns = RequiredSilverColumns - set(SilverDf.columns)
MissingMatchingColumns = RequiredMatchingColumns - set(MatchingDf.columns)

if MissingSilverColumns:
    raise ValueError(f"Missing Silver columns: {sorted(MissingSilverColumns)}")

if MissingMatchingColumns:
    raise ValueError(f"Missing matching columns: {sorted(MissingMatchingColumns)}")

NullPairKeys = ApprovedMatchesDf.filter(
    F.col("StyleHaus_ProductID").isNull()
    | (F.col("StyleHaus_ProductID") == "")
    | F.col("UrbanThread_ProductID").isNull()
    | (F.col("UrbanThread_ProductID") == "")
).count()

if NullPairKeys:
    raise ValueError(f"Found {NullPairKeys} approved pairs with null/empty product IDs.")

StyleHausConflictsDf = (
    ApprovedMatchesDf
    .groupBy("StyleHaus_ProductID")
    .agg(F.countDistinct("UrbanThread_ProductID").alias("AssignmentCount"))
    .filter(F.col("AssignmentCount") > 1)
)

UrbanThreadConflictsDf = (
    ApprovedMatchesDf
    .groupBy("UrbanThread_ProductID")
    .agg(F.countDistinct("StyleHaus_ProductID").alias("AssignmentCount"))
    .filter(F.col("AssignmentCount") > 1)
)

StyleHausConflictCount = StyleHausConflictsDf.count()
UrbanThreadConflictCount = UrbanThreadConflictsDf.count()

if StyleHausConflictCount or UrbanThreadConflictCount:
    display(StyleHausConflictsDf)
    display(UrbanThreadConflictsDf)
    raise ValueError(
        "Approved matches are not one-to-one. "
        f"StyleHaus conflicts={StyleHausConflictCount}; "
        f"UrbanThread conflicts={UrbanThreadConflictCount}."
    )

print("Approved pairs passed one-to-one and key checks.")

StatementMeta(, 650a0266-f6d3-41c0-bb82-b95dff4eb5a1, 8, Finished, Available, Finished, False)

Approved pairs passed one-to-one and key checks.


In [7]:
# Cell 4 – Verify approved match IDs exist in the correct Silver source

StyleHausSilverIdsDf = (
    SilverDf
    .filter(F.col("SourceSystem") == "StyleHaus")
    .select(F.col("SourceProductId").alias("StyleHaus_ProductID"))
    .distinct()
)

UrbanThreadSilverIdsDf = (
    SilverDf
    .filter(F.col("SourceSystem") == "UrbanThread")
    .select(F.col("SourceProductId").alias("UrbanThread_ProductID"))
    .distinct()
)

MissingStyleHausMatchesDf = ApprovedMatchesDf.join(
    StyleHausSilverIdsDf,
    on="StyleHaus_ProductID",
    how="left_anti"
)

MissingUrbanThreadMatchesDf = ApprovedMatchesDf.join(
    UrbanThreadSilverIdsDf,
    on="UrbanThread_ProductID",
    how="left_anti"
)

MissingStyleHausCount = MissingStyleHausMatchesDf.count()
MissingUrbanThreadCount = MissingUrbanThreadMatchesDf.count()

if MissingStyleHausCount or MissingUrbanThreadCount:
    print("Approved StyleHaus IDs missing from Silver:")
    display(MissingStyleHausMatchesDf)
    print("Approved UrbanThread IDs missing from Silver:")
    display(MissingUrbanThreadMatchesDf)
    raise ValueError(
        "Some approved matching IDs do not exist in their corresponding Silver source. "
        f"StyleHaus missing={MissingStyleHausCount}; "
        f"UrbanThread missing={MissingUrbanThreadCount}."
    )

print("All approved matching IDs exist in Silver.")

StatementMeta(, 650a0266-f6d3-41c0-bb82-b95dff4eb5a1, 9, Finished, Available, Finished, False)

All approved matching IDs exist in Silver.


In [8]:
# Cell 5 – Generate deterministic group IDs for matched pairs

MatchedGroupsDf = (
    ApprovedMatchesDf
    .withColumn(
        "GoldProductGroupId",
        F.concat(
            F.lit("GPG-"),
            F.sha2(
                F.concat_ws(
                    "|",
                    F.array_sort(
                        F.array(
                            F.concat(
                                F.lit("StyleHaus:"),
                                F.col("StyleHaus_ProductID")
                            ),
                            F.concat(
                                F.lit("UrbanThread:"),
                                F.col("UrbanThread_ProductID")
                            )
                        )
                    )
                ),
                256
            )
        )
    )
)

display(MatchedGroupsDf)

StatementMeta(, 650a0266-f6d3-41c0-bb82-b95dff4eb5a1, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 142bf9c2-9a29-41bc-b031-5578a2e3d4a6)

In [9]:
# Cell 6 – Create one group-member row for each side of every matched pair

MatchedStyleHausMembersDf = MatchedGroupsDf.select(
    "GoldProductGroupId",
    F.lit("StyleHaus").alias("SourceSystem"),
    F.col("StyleHaus_ProductID").alias("SourceProductId"),
    F.lit("MATCHED").alias("GroupType")
)

MatchedUrbanThreadMembersDf = MatchedGroupsDf.select(
    "GoldProductGroupId",
    F.lit("UrbanThread").alias("SourceSystem"),
    F.col("UrbanThread_ProductID").alias("SourceProductId"),
    F.lit("MATCHED").alias("GroupType")
)

MatchedMembersDf = (
    MatchedStyleHausMembersDf
    .unionByName(MatchedUrbanThreadMembersDf)
)

print("Matched group-member rows:", MatchedMembersDf.count())
display(MatchedMembersDf)

StatementMeta(, 650a0266-f6d3-41c0-bb82-b95dff4eb5a1, 11, Finished, Available, Finished, False)

Matched group-member rows: 44


SynapseWidget(Synapse.DataFrame, 778a8aa7-2d71-4cee-8df8-8a36a822bac2)

In [10]:
# Cell 7 – Find all Silver products not included in an approved MATCH pair

MatchedStyleHausIdsDf = ApprovedMatchesDf.select(
    F.col("StyleHaus_ProductID").alias("SourceProductId")
).distinct()

MatchedUrbanThreadIdsDf = ApprovedMatchesDf.select(
    F.col("UrbanThread_ProductID").alias("SourceProductId")
).distinct()

UnmatchedStyleHausDf = (
    SilverDf
    .filter(F.col("SourceSystem") == "StyleHaus")
    .join(MatchedStyleHausIdsDf, on="SourceProductId", how="left_anti")
)

UnmatchedUrbanThreadDf = (
    SilverDf
    .filter(F.col("SourceSystem") == "UrbanThread")
    .join(MatchedUrbanThreadIdsDf, on="SourceProductId", how="left_anti")
)

UnmatchedProductsDf = UnmatchedStyleHausDf.unionByName(UnmatchedUrbanThreadDf)

print("Unmatched individual products:", UnmatchedProductsDf.count())

StatementMeta(, 650a0266-f6d3-41c0-bb82-b95dff4eb5a1, 12, Finished, Available, Finished, False)

Unmatched individual products: 120


In [11]:
# Cell 8 – Assign a deterministic individual group to every unmatched product

UnmatchedGroupsDf = UnmatchedProductsDf.select(
    F.concat(
        F.lit("GPG-"),
        F.sha2(
            F.concat(
                F.col("SourceSystem"),
                F.lit(":"),
                F.col("SourceProductId")
            ),
            256
        )
    ).alias("GoldProductGroupId"),
    F.col("SourceSystem"),
    F.col("SourceProductId"),
    F.lit("UNMATCHED").alias("GroupType")
)

display(UnmatchedGroupsDf.limit(20))

StatementMeta(, 650a0266-f6d3-41c0-bb82-b95dff4eb5a1, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d8196253-bb2f-45f6-ba15-8257585b96d6)

In [12]:
# Cell 9 – Build the complete product-to-group mapping

ProductGroupMembersDf = (
    MatchedMembersDf
    .unionByName(UnmatchedGroupsDf)
    .select(
        "GoldProductGroupId",
        "SourceSystem",
        "SourceProductId",
        "GroupType"
    )
)

print("Total Silver products:", SilverDf.count())
print("Total group-member rows:", ProductGroupMembersDf.count())
print(
    "Total distinct product groups:",
    ProductGroupMembersDf.select("GoldProductGroupId").distinct().count()
)

display(
    ProductGroupMembersDf
    .orderBy("GoldProductGroupId", "SourceSystem")
)

StatementMeta(, 650a0266-f6d3-41c0-bb82-b95dff4eb5a1, 14, Finished, Available, Finished, False)

Total Silver products: 164
Total group-member rows: 164
Total distinct product groups: 142


SynapseWidget(Synapse.DataFrame, f3d09990-b0a5-4e9f-a454-f5f0b8508838)

## Final cell – Reconciliation for this sub-phase

For a complete one-to-one match set:
- Every Silver product must appear exactly once in `ProductGroupMembersDf`.
- Every approved pair contributes two member rows but one group.
- Every unmatched product contributes one member row and one group.
- Therefore, expected group count = Silver product count − approved pair count.

The cell below checks these properties and stops if the mapping is incomplete or contains duplicate source-product membership.

In [13]:
# Cell 10 – Grouping reconciliation

SilverCount = SilverDf.count()
ApprovedPairCount = ApprovedMatchesDf.count()
GroupMemberCount = ProductGroupMembersDf.count()
DistinctGroupCount = ProductGroupMembersDf.select("GoldProductGroupId").distinct().count()

DuplicateMembershipDf = (
    ProductGroupMembersDf
    .groupBy("SourceSystem", "SourceProductId")
    .count()
    .filter(F.col("count") != 1)
)

DuplicateMembershipCount = DuplicateMembershipDf.count()

MissingSilverProductsDf = (
    SilverDf
    .select("SourceSystem", "SourceProductId")
    .join(
        ProductGroupMembersDf.select("SourceSystem", "SourceProductId"),
        on=["SourceSystem", "SourceProductId"],
        how="left_anti"
    )
)

MissingSilverCount = MissingSilverProductsDf.count()

ExpectedGroupCount = SilverCount - ApprovedPairCount

print("Silver product count:", SilverCount)
print("Approved pair count:", ApprovedPairCount)
print("Expected group count:", ExpectedGroupCount)
print("Actual group-member rows:", GroupMemberCount)
print("Actual distinct group count:", DistinctGroupCount)
print("Duplicate source-product memberships:", DuplicateMembershipCount)
print("Silver products missing from grouping:", MissingSilverCount)

if GroupMemberCount != SilverCount:
    raise ValueError("Group-member row count does not reconcile with Silver.")

if DistinctGroupCount != ExpectedGroupCount:
    raise ValueError(
        f"Group count mismatch. Expected {ExpectedGroupCount}, "
        f"found {DistinctGroupCount}."
    )

if DuplicateMembershipCount:
    display(DuplicateMembershipDf)
    raise ValueError("A Silver source product appears more than once in the grouping map.")

if MissingSilverCount:
    display(MissingSilverProductsDf)
    raise ValueError("Some Silver products are missing from the grouping map.")

print("PHASE 4.2 GROUPING RECONCILIATION PASSED.")

StatementMeta(, 650a0266-f6d3-41c0-bb82-b95dff4eb5a1, 15, Finished, Available, Finished, False)

Silver product count: 164
Approved pair count: 22
Expected group count: 142
Actual group-member rows: 164
Actual distinct group count: 142
Duplicate source-product memberships: 0
Silver products missing from grouping: 0
PHASE 4.2 GROUPING RECONCILIATION PASSED.


In [14]:
ProductGroupMembersDf.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("semantic.productgroupmembers")

StatementMeta(, 650a0266-f6d3-41c0-bb82-b95dff4eb5a1, 16, Finished, Available, Finished, False)